# 06 — Generation Metrics

Questo notebook valuta le risposte prodotte per il task
`grounded_answer_generation`.

La valutazione automatica comprende:

- Sentence BLEU con smoothing;
- Corpus BLEU;
- ROUGE-1 precision, recall e F1;
- ROUGE-2 precision, recall e F1;
- ROUGE-L precision, recall e F1;
- BERTScore precision, recall e F1;
- lunghezza della reference;
- lunghezza della risposta generata;
- rapporto tra la lunghezza generata e quella della reference.

Il notebook produce inoltre un file Excel contenente le risposte generative
e due colonne vuote:

- `valutazione_A`;
- `valutazione_B`.

Le valutazioni soggettive non vengono elaborate in questo notebook.
Saranno importate e utilizzate successivamente nel notebook dedicato ai
confronti e ai ranking.

Il notebook non:

- confronta gli adapter con la baseline;
- assegna posizioni in classifica;
- costruisce un ranking complessivo;
- combina metriche automatiche e valutazioni soggettive.

Cella 2 — Installazione delle dipendenze

Eseguila la prima volta o quando le librerie non sono già disponibili.

%pip install -q nltk rouge-score bert-score xlsxwriter

Dopo l’installazione, se Jupyter lo richiede, riavvia il kernel e riparti dalla cella successiva.

In [2]:
from pathlib import Path
from importlib.metadata import (
    PackageNotFoundError,
    version,
)

import gc
import json
import re

import numpy as np
import pandas as pd
import torch

from bert_score import BERTScorer
from nltk.translate.bleu_score import (
    SmoothingFunction,
    corpus_bleu,
    sentence_bleu,
)
from rouge_score import rouge_scorer

C:\Users\fede2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Modificare esclusivamente questo percorso se il progetto si trova altrove.
PROJECT_ROOT = Path(
    r"C:\Users\fede2\Desktop\Project"
).resolve()


INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "metrics"
    / "generation_inputs_for_06.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "metrics"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


OUTPUT_BY_EXAMPLE_PATH = (
    OUTPUT_DIR
    / "generation_metrics_by_example.csv"
)

OUTPUT_BY_MODEL_PATH = (
    OUTPUT_DIR
    / "generation_metrics_by_model.csv"
)

SUBJECTIVE_EXCEL_PATH = (
    OUTPUT_DIR
    / "subjective_evaluation_to_complete.xlsx"
)

METADATA_PATH = (
    OUTPUT_DIR
    / "generation_metrics_metadata.json"
)


MODEL_ORDER = [
    "base",
    "regulation",
    "positive_legends",
    "negative_legends",
    "negative_to_correction",
]

EXPECTED_TASK_NAME = (
    "grounded_answer_generation"
)

EXPECTED_EXAMPLES_PER_MODEL = 20

EXPECTED_TOTAL_EXAMPLES = (
    len(MODEL_ORDER)
    * EXPECTED_EXAMPLES_PER_MODEL
)


# BLEU-4 standard: unigrammi, bigrammi, trigrammi e 4-grammi
# contribuiscono con lo stesso peso.
BLEU_WEIGHTS = (
    0.25,
    0.25,
    0.25,
    0.25,
)

# Lo smoothing evita che il Sentence BLEU diventi automaticamente zero
# quando una risposta non presenta sovrapposizioni per un certo ordine
# di n-grammi.
BLEU_SMOOTHING_METHOD = (
    SmoothingFunction().method1
)


# BERTScore utilizzerà la GPU quando disponibile.
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# Ridurre a 4 o 2 soltanto in caso di memoria GPU insufficiente.
BERTSCORE_BATCH_SIZE = 8


print("Input:", INPUT_PATH)
print("Output directory:", OUTPUT_DIR)
print("Dispositivo BERTScore:", DEVICE)
print("Risposte attese:", EXPECTED_TOTAL_EXAMPLES)

Input: C:\Users\fede2\Desktop\Project\data\metrics\generation_inputs_for_06.csv
Output directory: C:\Users\fede2\Desktop\Project\data\metrics
Dispositivo BERTScore: cuda
Risposte attese: 100


In [4]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"File non trovato:\n{INPUT_PATH}\n\n"
        "Eseguire prima il notebook 05."
    )


generation_df = pd.read_csv(
    INPUT_PATH
)


print(
    "Righe caricate:",
    len(generation_df),
)

display(
    generation_df.head()
)

Righe caricate: 100


,model_name,task_name,example_id,reference_answer,generated_answer,reference_word_count,generated_word_count,word_length_ratio
0,base,grounded_answer_generation,GEN_001,The entity must pursue the objective of achiev...,The corporate entity must pursue an objective ...,29,34,1.172414
1,base,grounded_answer_generation,GEN_002,The committee must give priority to the candid...,The selection committee must select candidates...,23,39,1.695652
2,base,grounded_answer_generation,GEN_003,The enterprise is not legally bound by the tar...,"No, a small enterprise with 50 employees is no...",24,38,1.583333
3,base,grounded_answer_generation,GEN_004,The organization should redesign succession pl...,Organizations should establish transparent and...,23,44,1.913043
4,base,grounded_answer_generation,GEN_005,"The organization must establish clear, neutral...",The organization must ensure that its board-se...,21,41,1.952381


In [5]:
REQUIRED_COLUMNS = {
    "model_name",
    "task_name",
    "example_id",
    "reference_answer",
    "generated_answer",
}


missing_columns = (
    REQUIRED_COLUMNS
    - set(generation_df.columns)
)

if missing_columns:
    raise RuntimeError(
        "Nel file mancano le colonne obbligatorie: "
        + ", ".join(
            sorted(missing_columns)
        )
    )


# Normalizzazione minima dei campi testuali.
for column_name in [
    "model_name",
    "task_name",
    "example_id",
    "reference_answer",
    "generated_answer",
]:
    generation_df[column_name] = (
        generation_df[column_name]
        .fillna("")
        .astype(str)
        .str.strip()
    )


unknown_models = (
    set(generation_df["model_name"])
    - set(MODEL_ORDER)
)

if unknown_models:
    raise RuntimeError(
        "Sono presenti modelli non configurati: "
        + ", ".join(
            sorted(unknown_models)
        )
    )


missing_models = (
    set(MODEL_ORDER)
    - set(generation_df["model_name"])
)

if missing_models:
    raise RuntimeError(
        "Mancano i seguenti modelli: "
        + ", ".join(
            sorted(missing_models)
        )
    )


unexpected_tasks = (
    set(generation_df["task_name"])
    - {EXPECTED_TASK_NAME}
)

if unexpected_tasks:
    raise RuntimeError(
        "Il file contiene task diversi da "
        f"{EXPECTED_TASK_NAME}: "
        + ", ".join(
            sorted(unexpected_tasks)
        )
    )


duplicate_mask = generation_df.duplicated(
    subset=[
        "model_name",
        "example_id",
    ],
    keep=False,
)

if duplicate_mask.any():
    display(
        generation_df.loc[
            duplicate_mask,
            [
                "model_name",
                "example_id",
            ],
        ]
    )

    raise RuntimeError(
        "Sono presenti coppie duplicate "
        "(model_name, example_id)."
    )


empty_reference_mask = (
    generation_df["reference_answer"]
    == ""
)

if empty_reference_mask.any():
    display(
        generation_df.loc[
            empty_reference_mask,
            [
                "model_name",
                "example_id",
                "reference_answer",
            ],
        ]
    )

    raise RuntimeError(
        "Sono presenti reference vuote."
    )


empty_generated_mask = (
    generation_df["generated_answer"]
    == ""
)

if empty_generated_mask.any():
    display(
        generation_df.loc[
            empty_generated_mask,
            [
                "model_name",
                "example_id",
                "generated_answer",
            ],
        ]
    )

    raise RuntimeError(
        "Sono presenti risposte generate vuote."
    )


counts_by_model = (
    generation_df
    .groupby("model_name")
    .size()
    .reindex(
        MODEL_ORDER,
        fill_value=0,
    )
)

display(
    counts_by_model.rename(
        "n_examples"
    ).to_frame()
)


incorrect_counts = (
    counts_by_model[
        counts_by_model
        != EXPECTED_EXAMPLES_PER_MODEL
    ]
)

if not incorrect_counts.empty:
    raise RuntimeError(
        "Ogni modello deve avere esattamente "
        f"{EXPECTED_EXAMPLES_PER_MODEL} risposte.\n"
        + incorrect_counts.to_string()
    )


if (
    len(generation_df)
    != EXPECTED_TOTAL_EXAMPLES
):
    raise RuntimeError(
        f"Numero di righe trovato: "
        f"{len(generation_df)}.\n"
        f"Numero atteso: "
        f"{EXPECTED_TOTAL_EXAMPLES}."
    )


# Ogni example_id deve avere la stessa reference per tutti i modelli.
reference_consistency = (
    generation_df
    .groupby("example_id")
    ["reference_answer"]
    .nunique()
)

inconsistent_references = (
    reference_consistency[
        reference_consistency != 1
    ]
)

if not inconsistent_references.empty:
    display(
        inconsistent_references
    )

    raise RuntimeError(
        "Alcuni example_id presentano reference "
        "diverse tra i modelli."
    )


# Ogni example_id deve comparire una volta per ciascun modello.
model_coverage = (
    generation_df
    .groupby("example_id")
    ["model_name"]
    .nunique()
)

incomplete_examples = (
    model_coverage[
        model_coverage
        != len(MODEL_ORDER)
    ]
)

if not incomplete_examples.empty:
    display(
        incomplete_examples
    )

    raise RuntimeError(
        "Alcuni example_id non sono presenti "
        "per tutti i modelli."
    )


print("Validazione completata correttamente.")

,n_examples
model_name,
base,20
regulation,20
positive_legends,20
negative_legends,20
negative_to_correction,20


Validazione completata correttamente.


## BLEU e ROUGE

### BLEU

Vengono calcolati due risultati:

- `sentence_bleu`: un punteggio per ciascuna risposta;
- `corpus_bleu`: un punteggio complessivo per ciascun modello.

Il Sentence BLEU utilizza lo smoothing, perché sulle singole risposte
l'assenza di alcuni n-grammi potrebbe altrimenti produrre automaticamente
un valore pari a zero.

### ROUGE

Per ogni risposta vengono calcolati:

- ROUGE-1 precision, recall e F1;
- ROUGE-2 precision, recall e F1;
- ROUGE-L precision, recall e F1.

Lo stemming viene disabilitato perché l'implementazione utilizzata adotta
uno stemmer pensato principalmente per l'inglese, mentre il dataset è in
italiano.

In [6]:
# Tokenizzazione semplice e riproducibile.
#
# Manteniamo lettere e numeri, convertendo il testo in minuscolo.
# Non richiede il download di risorse aggiuntive di NLTK.
BLEU_TOKEN_PATTERN = re.compile(
    r"\b\w+\b",
    flags=re.UNICODE,
)


def tokenize_for_bleu(
    text: str,
) -> list[str]:
    return BLEU_TOKEN_PATTERN.findall(
        str(text).casefold()
    )


rouge_evaluator = rouge_scorer.RougeScorer(
    [
        "rouge1",
        "rouge2",
        "rougeL",
    ],
    use_stemmer=False,
)

In [7]:
automatic_rows = []


for row in generation_df.itertuples(
    index=False
):
    reference_tokens = tokenize_for_bleu(
        row.reference_answer
    )

    generated_tokens = tokenize_for_bleu(
        row.generated_answer
    )


    sentence_bleu_score = float(
        sentence_bleu(
            references=[
                reference_tokens
            ],
            hypothesis=generated_tokens,
            weights=BLEU_WEIGHTS,
            smoothing_function=(
                BLEU_SMOOTHING_METHOD
            ),
        )
    )


    rouge_scores = rouge_evaluator.score(
        target=row.reference_answer,
        prediction=row.generated_answer,
    )


    automatic_rows.append({
        "model_name": row.model_name,
        "task_name": row.task_name,
        "example_id": row.example_id,

        "reference_answer": (
            row.reference_answer
        ),

        "generated_answer": (
            row.generated_answer
        ),

        "sentence_bleu": (
            sentence_bleu_score
        ),

        "rouge1_precision": float(
            rouge_scores[
                "rouge1"
            ].precision
        ),

        "rouge1_recall": float(
            rouge_scores[
                "rouge1"
            ].recall
        ),

        "rouge1_f1": float(
            rouge_scores[
                "rouge1"
            ].fmeasure
        ),

        "rouge2_precision": float(
            rouge_scores[
                "rouge2"
            ].precision
        ),

        "rouge2_recall": float(
            rouge_scores[
                "rouge2"
            ].recall
        ),

        "rouge2_f1": float(
            rouge_scores[
                "rouge2"
            ].fmeasure
        ),

        "rougeL_precision": float(
            rouge_scores[
                "rougeL"
            ].precision
        ),

        "rougeL_recall": float(
            rouge_scores[
                "rougeL"
            ].recall
        ),

        "rougeL_f1": float(
            rouge_scores[
                "rougeL"
            ].fmeasure
        ),
    })


generation_metrics_df = pd.DataFrame(
    automatic_rows
)


display(
    generation_metrics_df[
        [
            "model_name",
            "example_id",
            "sentence_bleu",
            "rouge1_f1",
            "rouge2_f1",
            "rougeL_f1",
        ]
    ]
    .head(10)
    .round(4)
)

,model_name,example_id,sentence_bleu,rouge1_f1,rouge2_f1,rougeL_f1
0,base,GEN_001,0.1190,0.4308,0.1905,0.3077
1,base,GEN_002,0.0146,0.2857,0.0328,0.2222
2,base,GEN_003,0.0926,0.4127,0.1639,0.3810
3,base,GEN_004,0.0064,0.1765,0.0000,0.0882
4,base,GEN_005,0.0294,0.3125,0.0645,0.1875
5,base,GEN_006,0.0266,0.2885,0.1176,0.2308
6,base,GEN_007,0.0462,0.3111,0.0682,0.2444
7,base,GEN_008,0.0145,0.1791,0.0615,0.1493
8,base,GEN_009,0.0203,0.2381,0.0488,0.1905
9,base,GEN_010,0.0091,0.3478,0.0000,0.2029


## BERTScore

BERTScore confronta semanticamente la risposta generata con la reference
utilizzando rappresentazioni contestuali dei token.

Per ciascuna risposta vengono calcolati:

- BERTScore precision;
- BERTScore recall;
- BERTScore F1.

La lingua viene impostata esplicitamente su italiano (`it`).

Non viene applicata la ponderazione IDF e non viene effettuato il rescaling
con baseline. La configurazione utilizzata viene salvata nel file di
metadati per rendere il risultato riproducibile.

Il primo avvio può richiedere il download del modello multilingue utilizzato
da BERTScore.

In [8]:
# Libera eventuale memoria non più utilizzata prima di caricare
# il modello richiesto da BERTScore.
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


bert_scorer = BERTScorer(
    lang="it",
    batch_size=BERTSCORE_BATCH_SIZE,
    device=DEVICE,
    idf=False,
    rescale_with_baseline=False,
)


generated_answers = (
    generation_metrics_df[
        "generated_answer"
    ]
    .tolist()
)

reference_answers = (
    generation_metrics_df[
        "reference_answer"
    ]
    .tolist()
)


bert_precision, bert_recall, bert_f1 = (
    bert_scorer.score(
        generated_answers,
        reference_answers,
    )
)


generation_metrics_df[
    "bertscore_precision"
] = (
    bert_precision
    .detach()
    .cpu()
    .numpy()
    .astype(float)
)

generation_metrics_df[
    "bertscore_recall"
] = (
    bert_recall
    .detach()
    .cpu()
    .numpy()
    .astype(float)
)

generation_metrics_df[
    "bertscore_f1"
] = (
    bert_f1
    .detach()
    .cpu()
    .numpy()
    .astype(float)
)


BERTSCORE_HASH = (
    bert_scorer.hash
)


print(
    "Configurazione BERTScore:",
    BERTSCORE_HASH,
)


display(
    generation_metrics_df[
        [
            "model_name",
            "example_id",
            "bertscore_precision",
            "bertscore_recall",
            "bertscore_f1",
        ]
    ]
    .head(10)
    .round(4)
)

C:\Users\fede2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fede2\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100

Configurazione BERTScore: bert-base-multilingual-cased_L9_no-idf_version=0.3.12(hug_trans=5.13.1)


,model_name,example_id,bertscore_precision,bertscore_recall,bertscore_f1
0,base,GEN_001,0.8126,0.8179,0.8152
1,base,GEN_002,0.6992,0.7030,0.7011
2,base,GEN_003,0.7288,0.7182,0.7235
3,base,GEN_004,0.7315,0.7529,0.7420
4,base,GEN_005,0.7163,0.7231,0.7197
5,base,GEN_006,0.7110,0.7866,0.7469
6,base,GEN_007,0.7033,0.7969,0.7472
7,base,GEN_008,0.7518,0.7859,0.7684
8,base,GEN_009,0.7293,0.7420,0.7356
9,base,GEN_010,0.7269,0.7381,0.7324


In [9]:
generation_metrics_df[
    "reference_word_count"
] = (
    generation_metrics_df[
        "reference_answer"
    ]
    .str.split()
    .str.len()
    .astype(int)
)

generation_metrics_df[
    "generated_word_count"
] = (
    generation_metrics_df[
        "generated_answer"
    ]
    .str.split()
    .str.len()
    .astype(int)
)

generation_metrics_df[
    "word_length_ratio"
] = (
    generation_metrics_df[
        "generated_word_count"
    ]
    /
    generation_metrics_df[
        "reference_word_count"
    ]
)


display(
    generation_metrics_df[
        [
            "model_name",
            "example_id",
            "reference_word_count",
            "generated_word_count",
            "word_length_ratio",
        ]
    ]
    .head(10)
    .round(4)
)

,model_name,example_id,reference_word_count,generated_word_count,word_length_ratio
0,base,GEN_001,29,34,1.1724
1,base,GEN_002,23,39,1.6957
2,base,GEN_003,24,38,1.5833
3,base,GEN_004,23,44,1.9130
4,base,GEN_005,21,41,1.9524
5,base,GEN_006,25,78,3.1200
6,base,GEN_007,23,64,2.7826
7,base,GEN_008,23,42,1.8261
8,base,GEN_009,24,60,2.5000
9,base,GEN_010,28,37,1.3214


In [10]:
corpus_bleu_rows = []


for model_name in MODEL_ORDER:
    model_group = generation_metrics_df[
        generation_metrics_df[
            "model_name"
        ]
        == model_name
    ]


    list_of_references = [
        [
            tokenize_for_bleu(
                reference
            )
        ]
        for reference in model_group[
            "reference_answer"
        ]
    ]

    hypotheses = [
        tokenize_for_bleu(
            generated
        )
        for generated in model_group[
            "generated_answer"
        ]
    ]


    model_corpus_bleu = float(
        corpus_bleu(
            list_of_references=(
                list_of_references
            ),
            hypotheses=hypotheses,
            weights=BLEU_WEIGHTS,
            smoothing_function=(
                BLEU_SMOOTHING_METHOD
            ),
        )
    )


    corpus_bleu_rows.append({
        "model_name": model_name,
        "corpus_bleu": (
            model_corpus_bleu
        ),
    })


corpus_bleu_df = pd.DataFrame(
    corpus_bleu_rows
)


display(
    corpus_bleu_df.round(4)
)

,model_name,corpus_bleu
0,base,0.0319
1,regulation,0.0450
2,positive_legends,0.0758
3,negative_legends,0.0609
4,negative_to_correction,0.0491


In [11]:
generation_metrics_by_model_df = (
    generation_metrics_df
    .groupby(
        "model_name",
        as_index=False,
    )
    .agg(
        n_examples=(
            "example_id",
            "count",
        ),

        mean_sentence_bleu=(
            "sentence_bleu",
            "mean",
        ),

        std_sentence_bleu=(
            "sentence_bleu",
            "std",
        ),

        mean_rouge1_precision=(
            "rouge1_precision",
            "mean",
        ),

        mean_rouge1_recall=(
            "rouge1_recall",
            "mean",
        ),

        mean_rouge1_f1=(
            "rouge1_f1",
            "mean",
        ),

        std_rouge1_f1=(
            "rouge1_f1",
            "std",
        ),

        mean_rouge2_precision=(
            "rouge2_precision",
            "mean",
        ),

        mean_rouge2_recall=(
            "rouge2_recall",
            "mean",
        ),

        mean_rouge2_f1=(
            "rouge2_f1",
            "mean",
        ),

        std_rouge2_f1=(
            "rouge2_f1",
            "std",
        ),

        mean_rougeL_precision=(
            "rougeL_precision",
            "mean",
        ),

        mean_rougeL_recall=(
            "rougeL_recall",
            "mean",
        ),

        mean_rougeL_f1=(
            "rougeL_f1",
            "mean",
        ),

        std_rougeL_f1=(
            "rougeL_f1",
            "std",
        ),

        mean_bertscore_precision=(
            "bertscore_precision",
            "mean",
        ),

        mean_bertscore_recall=(
            "bertscore_recall",
            "mean",
        ),

        mean_bertscore_f1=(
            "bertscore_f1",
            "mean",
        ),

        std_bertscore_f1=(
            "bertscore_f1",
            "std",
        ),

        mean_reference_word_count=(
            "reference_word_count",
            "mean",
        ),

        mean_generated_word_count=(
            "generated_word_count",
            "mean",
        ),

        median_generated_word_count=(
            "generated_word_count",
            "median",
        ),

        mean_word_length_ratio=(
            "word_length_ratio",
            "mean",
        ),
    )
)


generation_metrics_by_model_df = (
    generation_metrics_by_model_df
    .merge(
        corpus_bleu_df,
        on="model_name",
        how="left",
        validate="one_to_one",
    )
)


model_order_map = {
    model_name: index
    for index, model_name in enumerate(
        MODEL_ORDER
    )
}

generation_metrics_by_model_df[
    "_model_order"
] = (
    generation_metrics_by_model_df[
        "model_name"
    ]
    .map(model_order_map)
)

generation_metrics_by_model_df = (
    generation_metrics_by_model_df
    .sort_values(
        "_model_order"
    )
    .drop(
        columns="_model_order"
    )
    .reset_index(
        drop=True
    )
)


summary_columns = [
    "model_name",
    "n_examples",
    "mean_sentence_bleu",
    "corpus_bleu",
    "mean_rouge1_f1",
    "mean_rouge2_f1",
    "mean_rougeL_f1",
    "mean_bertscore_f1",
    "mean_generated_word_count",
    "mean_word_length_ratio",
]


display(
    generation_metrics_by_model_df[
        summary_columns
    ]
    .round(4)
)

,model_name,n_examples,mean_sentence_bleu,corpus_bleu,mean_rouge1_f1,mean_rouge2_f1,mean_rougeL_f1,mean_bertscore_f1,mean_generated_word_count,mean_word_length_ratio
0,base,20,0.0312,0.0319,0.2590,0.0702,0.1881,0.7369,48.65,2.0518
1,regulation,20,0.0356,0.0450,0.2762,0.0777,0.2355,0.7465,22.60,0.9731
2,positive_legends,20,0.0592,0.0758,0.3145,0.1224,0.2855,0.7658,18.75,0.7937
3,negative_legends,20,0.0495,0.0609,0.2940,0.1018,0.2603,0.7581,18.05,0.7628
4,negative_to_correction,20,0.0381,0.0491,0.2851,0.0870,0.2300,0.7647,19.90,0.8460


## Excel per la valutazione soggettiva

Il notebook genera un nuovo file Excel copiando le colonne presenti nel file
`generation_inputs_for_06.csv` e aggiungendo esclusivamente:

- `valutazione_A`;
- `valutazione_B`.

Le righe non vengono:

- randomizzate;
- anonimizzate;
- riordinate.

Il file non contiene formule, menu, fogli aggiuntivi o calcoli automatici.

Le due colonne potranno essere compilate successivamente con valori da 0 a 5.
I punteggi saranno elaborati nel notebook dedicato ai confronti e ai ranking.

In [12]:
subjective_evaluation_df = (
    generation_df.copy()
)


subjective_evaluation_df[
    "valutazione_A"
] = pd.NA

subjective_evaluation_df[
    "valutazione_B"
] = pd.NA


subjective_evaluation_df.to_excel(
    SUBJECTIVE_EXCEL_PATH,
    index=False,
    sheet_name="Valutazioni",
    engine="xlsxwriter",
)


print(
    "Excel creato:",
    SUBJECTIVE_EXCEL_PATH,
)

display(
    subjective_evaluation_df.head()
)

Excel creato: C:\Users\fede2\Desktop\Project\data\metrics\subjective_evaluation_to_complete.xlsx


,model_name,task_name,example_id,reference_answer,generated_answer,reference_word_count,generated_word_count,word_length_ratio,valutazione_A,valutazione_B
0,base,grounded_answer_generation,GEN_001,The entity must pursue the objective of achiev...,The corporate entity must pursue an objective ...,29,34,1.172414,<NA>,<NA>
1,base,grounded_answer_generation,GEN_002,The committee must give priority to the candid...,The selection committee must select candidates...,23,39,1.695652,<NA>,<NA>
2,base,grounded_answer_generation,GEN_003,The enterprise is not legally bound by the tar...,"No, a small enterprise with 50 employees is no...",24,38,1.583333,<NA>,<NA>
3,base,grounded_answer_generation,GEN_004,The organization should redesign succession pl...,Organizations should establish transparent and...,23,44,1.913043,<NA>,<NA>
4,base,grounded_answer_generation,GEN_005,"The organization must establish clear, neutral...",The organization must ensure that its board-se...,21,41,1.952381,<NA>,<NA>


In [13]:
# Manteniamo l'ordine originario delle righe nel risultato per esempio.
generation_metrics_df.to_csv(
    OUTPUT_BY_EXAMPLE_PATH,
    index=False,
    encoding="utf-8-sig",
)


generation_metrics_by_model_df.to_csv(
    OUTPUT_BY_MODEL_PATH,
    index=False,
    encoding="utf-8-sig",
)


print(
    "Metriche per esempio:",
    OUTPUT_BY_EXAMPLE_PATH,
)

print(
    "Metriche per modello:",
    OUTPUT_BY_MODEL_PATH,
)

Metriche per esempio: C:\Users\fede2\Desktop\Project\data\metrics\generation_metrics_by_example.csv
Metriche per modello: C:\Users\fede2\Desktop\Project\data\metrics\generation_metrics_by_model.csv


In [14]:
def get_package_version(
    package_name: str,
) -> str | None:
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


metadata = {
    "input_path": str(
        INPUT_PATH
    ),

    "output_by_example_path": str(
        OUTPUT_BY_EXAMPLE_PATH
    ),

    "output_by_model_path": str(
        OUTPUT_BY_MODEL_PATH
    ),

    "subjective_excel_path": str(
        SUBJECTIVE_EXCEL_PATH
    ),

    "number_of_models": len(
        MODEL_ORDER
    ),

    "number_of_examples_per_model": (
        EXPECTED_EXAMPLES_PER_MODEL
    ),

    "total_number_of_examples": int(
        len(generation_metrics_df)
    ),

    "bleu": {
        "sentence_bleu": True,
        "corpus_bleu": True,
        "weights": list(
            BLEU_WEIGHTS
        ),
        "smoothing": (
            "SmoothingFunction.method1"
        ),
        "tokenization": (
            "Unicode word tokens, casefolded"
        ),
    },

    "rouge": {
        "metrics": [
            "rouge1",
            "rouge2",
            "rougeL",
        ],
        "use_stemmer": False,
    },

    "bertscore": {
        "language": "it",
        "device": DEVICE,
        "batch_size": (
            BERTSCORE_BATCH_SIZE
        ),
        "idf": False,
        "rescale_with_baseline": False,
        "hash": BERTSCORE_HASH,
    },

    "length_metrics": {
        "reference_word_count": True,
        "generated_word_count": True,
        "word_length_ratio": True,
        "absolute_word_difference": False,
    },

    "excluded_metrics": [
        "semantic_cosine_similarity",
        "normalized_exact_match",
        "absolute_word_difference",
    ],

    "package_versions": {
        "pandas": get_package_version(
            "pandas"
        ),
        "numpy": get_package_version(
            "numpy"
        ),
        "torch": get_package_version(
            "torch"
        ),
        "nltk": get_package_version(
            "nltk"
        ),
        "rouge-score": get_package_version(
            "rouge-score"
        ),
        "bert-score": get_package_version(
            "bert-score"
        ),
        "transformers": get_package_version(
            "transformers"
        ),
        "xlsxwriter": get_package_version(
            "xlsxwriter"
        ),
    },
}


with METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Metadati salvati:",
    METADATA_PATH,
)

Metadati salvati: C:\Users\fede2\Desktop\Project\data\metrics\generation_metrics_metadata.json


In [18]:
EXPECTED_EXAMPLE_METRIC_COLUMNS = {
    "model_name",
    "task_name",
    "example_id",
    "reference_answer",
    "generated_answer",

    "sentence_bleu",

    "rouge1_precision",
    "rouge1_recall",
    "rouge1_f1",

    "rouge2_precision",
    "rouge2_recall",
    "rouge2_f1",

    "rougeL_precision",
    "rougeL_recall",
    "rougeL_f1",

    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1",

    "reference_word_count",
    "generated_word_count",
    "word_length_ratio",
}


missing_metric_columns = (
    EXPECTED_EXAMPLE_METRIC_COLUMNS
    - set(generation_metrics_df.columns)
)

if missing_metric_columns:
    raise RuntimeError(
        "Mancano metriche nel risultato per esempio: "
        + ", ".join(
            sorted(missing_metric_columns)
        )
    )


if (
    len(generation_metrics_df)
    != EXPECTED_TOTAL_EXAMPLES
):
    raise RuntimeError(
        "Il file delle metriche per esempio "
        "non contiene tutte le risposte."
    )


if (
    len(generation_metrics_by_model_df)
    != len(MODEL_ORDER)
):
    raise RuntimeError(
        "Il file aggregato non contiene "
        "tutti i modelli."
    )


if generation_metrics_df[
    list(
        EXPECTED_EXAMPLE_METRIC_COLUMNS
        - {
            "model_name",
            "task_name",
            "example_id",
            "reference_answer",
            "generated_answer",
        }
    )
].isna().any().any():
    raise RuntimeError(
        "Sono presenti valori mancanti "
        "nelle metriche automatiche."
    )


expected_output_files = [
    OUTPUT_BY_EXAMPLE_PATH,
    OUTPUT_BY_MODEL_PATH,
    SUBJECTIVE_EXCEL_PATH,
    METADATA_PATH,
]

missing_output_files = [
    str(path)
    for path in expected_output_files
    if not path.exists()
]

if missing_output_files:
    raise RuntimeError(
        "File di output mancanti:\n- "
        + "\n- ".join(
            missing_output_files
        )
    )


excel_check_df = pd.read_excel(
    SUBJECTIVE_EXCEL_PATH,
    sheet_name="Valutazioni",
)


if (
    len(excel_check_df)
    != EXPECTED_TOTAL_EXAMPLES
):
    raise RuntimeError(
        "L'Excel soggettivo non contiene "
        "tutte le risposte."
    )


for evaluation_column in [
    "valutazione_A",
    "valutazione_B",
]:
    if evaluation_column not in excel_check_df.columns:
        raise RuntimeError(
            f"Nell'Excel manca la colonna {evaluation_column}."
        )


print("\n" + "=" * 90)
print("NOTEBOOK 06 COMPLETATO")
print("=" * 90)

print(
    "\nRisposte valutate automaticamente:",
    len(generation_metrics_df),
)

print(
    "Modelli aggregati:",
    len(generation_metrics_by_model_df),
)

print("\nFile prodotti:")

for path in expected_output_files:
    print(f"- {path}")

print(
    "\nLe valutazioni A e B non sono state "
    "elaborate in questo notebook."
)

print(
    "Saranno importate successivamente nel "
    "notebook dei confronti e dei ranking."
)


NOTEBOOK 06 COMPLETATO

Risposte valutate automaticamente: 100
Modelli aggregati: 5

File prodotti:
- C:\Users\fede2\Desktop\Project\data\metrics\generation_metrics_by_example.csv
- C:\Users\fede2\Desktop\Project\data\metrics\generation_metrics_by_model.csv
- C:\Users\fede2\Desktop\Project\data\metrics\subjective_evaluation_to_complete.xlsx
- C:\Users\fede2\Desktop\Project\data\metrics\generation_metrics_metadata.json

Le valutazioni A e B non sono state elaborate in questo notebook.
Saranno importate successivamente nel notebook dei confronti e dei ranking.


Output del notebook
data/metrics/
├── generation_metrics_by_example.csv
├── generation_metrics_by_model.csv
├── generation_metrics_metadata.json
└── subjective_evaluation_to_complete.xlsx

Il file Excel conterrà esattamente i dati del generativo più:

valutazione_A
valutazione_B